In [ ]:
import numpy as np
import pandas as pd

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from models.MLPipeline import *
from utils import ASSETS_DIR

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import confusion_matrix, make_scorer
from sklearn.model_selection import GridSearchCV
import time
from sklearn.model_selection import ParameterGrid
from sklearn.decomposition import PCA


In [ ]:
df = pd.read_parquet(ASSETS_DIR / 'final_df.parquet')

In [ ]:


# 1. Separiamo la matrice delle Feature (X) dal Target (y)
X = df.drop(columns=['TARGET'])
X.drop(columns=['LBDEVAL'], inplace=True, errors='ignore')
y = df['TARGET']

# 2. TRAIN-TEST SPLIT (80% Train, 20% Test)
# stratify=y è fondamentale per mantenere le stesse percentuali di malati nei due set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Buchi (NaN) iniziali in X_train: {X_train.isna().sum().sum()}")

# 3. CONFIGURAZIONE DEL KNN IMPUTER

imputer = KNNImputer(n_neighbors=7, weights='distance')

# 4. ADDESTRAMENTO E TRASFORMAZIONE
# Il modello "impara" le distribuzioni SOLO da X_train
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)

# Il modello applica quanto imparato su X_test (senza barare)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# 5. ARROTONDAMENTO PER DATI CLINICI DISCRETI
# Riportiamo le medie del KNN a numeri interi (es. 0.66 diventa 1.0)
X_train_imp = X_train_imp.round()
X_test_imp = X_test_imp.round()

print(f"Buchi (NaN) finali in X_train: {X_train_imp.isna().sum().sum()}")
print(f"Buchi (NaN) finali in X_test: {X_test_imp.isna().sum().sum()}")

In [ ]:
scaler = MinMaxScaler()

x_train_scaled = scaler.fit_transform(X_train_imp)
x_test_scaled = scaler.transform(X_test_imp)

In [ ]:
pca = PCA(n_components=0.90, random_state=42)

X_train_pca = pca.fit_transform(x_train_scaled)
X_test_pca = pca.transform(x_test_scaled)

print(f"Dimensioni originali: {x_train_scaled.shape[1]} features")
print(f"Dimensioni dopo PCA: {X_train_pca.shape[1]} componenti principali (spiegano il 90% della varianza)")

# 3. VISUALIZZAZIONE DELLA VARIANZA SPIEGATA
plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o', linestyle='--')
plt.title("Varianza Cumulativa Spiegata dalle Componenti Principali")
plt.xlabel("Numero di Componenti")
plt.ylabel("Varianza Spiegata")
plt.grid(True)
plt.show()

In [ ]:
svc_base = SVC(
       kernel="rbf",
       C=1.0,
       gamma="scale",
       decision_function_shape="ovo"
    )

#Trasforma svc in un modello probabilistico usando una sigmoide
model = CalibratedClassifierCV(
    estimator=svc_base, 
    method='sigmoid', 
    cv=3,
    n_jobs=-1
)

smote_enn = SMOTEENN(random_state=42, n_jobs=-1)
pipeline = ImbPipeline(steps=[
    ('smoteenn', smote_enn),
    ('model', model)
])


In [ ]:

y_true_bin, y_proba = train_model_evaluate(X=X_train_pca, y=y_train, model=pipeline)
generate_predictions_and_cm(X_train_pca, y_train, pipeline)
plot_reliability_diagram(y_true_bin, y_proba[:, 2], title='Reliability Diagram - Random Forest (Train Set)')

In [ ]:
# Griglia di ottimizzazione per SVC Probabilistico
param_grid_svc = {
    # 1. Il parametro di Regolarizzazione (C)
    # Valori bassi (0.1) creano margini morbidi (generalizza di più)
    # Valori alti (10) creano margini duri (si adatta perfettamente al Train Set, rischio overfitting)
    'model__estimator__C': [0.1, 1, 10],
    
    # 2. Il coefficiente del Kernel (Gamma)
    # Definisce quanto "lontano" arriva l'influenza di un singolo paziente.
    # 'scale' è l'adattamento automatico, i valori numerici forzano la precisione dell'isola.
    'model__estimator__gamma': ['scale', 'auto', 0.01, 0.1],
    
}

In [ ]:


print("Configurazione dell'architettura SVC + GridSearch...")



scoring_dict = {
    'f1_macro': 'f1_macro',
    'roc_auc_pairwise': scorer_auc_alz_dlb, 
    'clinical_cost': scorer_costo_clinico
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid_svc,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=3
)
print(f"Avvio Grid Search: {len(ParameterGrid(param_grid_svc))} combinazioni x 5 Folds...")
start_time = time.time()


grid_search.fit(X_train_pca, y_train)

end_time = time.time()
print(f"\nRicerca completata in {(end_time - start_time)/60:.2f} minuti.")

# 6. Estrazione dei risultati vincenti
print("\n=== RISULTATI GRID SEARCH XGBOOST ===")
print(f"Miglior F1-Score (Macro) in Cross-Validation: {grid_search.best_score_:.4f}")
print("Migliori Iperparametri trovati:")
for param, value in grid_search.best_params_.items():
    print(f" - {param.replace('model__', '')}: {value}")



In [ ]:
best_model = grid_search.best_estimator_
svc = best_model.named_steps["model"]
y_true_bin, y_proba = train_model_evaluate(X=X_train_pca, y=y_train, model=best_model)
generate_predictions_and_cm(X_train_pca, y_train, best_model)
plot_reliability_diagram(y_true_bin, y_proba[:, 2], title='Reliability Diagram - Random Forest (Train Set)')

In [ ]:
y_true_bin, y_proba, y_pred = train_model_evaluate(X_train_pca, y_train, best_model, use_cv=False, X_test=X_test_pca, y_test= y_test)

cm = confusion_matrix(y_test, y_pred)

etichette = ['Sani (0)', 'Alzheimer (1)', 'Lewy Body (2)']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=etichette, yticklabels=etichette,
            linewidths=1, linecolor='black')

plt.title("confusion amtrix", fontsize=14, pad=15)
plt.ylabel('Diagnosi Reale (Medico)', fontsize=12, fontweight='bold')
plt.xlabel(f'Previsione ({type(model).__name__})', fontsize=12, fontweight='bold')
plt.show()

print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred, target_names=etichette))

In [ ]:
import pickle



models_dir = ASSETS_DIR / "models"

if not models_dir.exists():
    models_dir.mkdir(parents=True, exist_ok=True)



with open( models_dir / "svc.pkl", "wb") as file:
    pickle.dump(svc, file)